<a href="https://colab.research.google.com/github/Ospino89/spotify-dwh/blob/feature%2Fetl-data-endpoints/notebooks/eda_spotify_ivan_ospino_darcy_escalante_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de Reproducciones en Spotify

> **Universidad de Pamplona · Bases de Datos II · 2026-I**  
> **Profesor:** Juan Alejandro Carrillo Jaimes  
> **Integrantes:** Ivan Ospino · Darcy Escalante

---

| Sección | Contenido |
|---|---|
| 1 | Instalación y configuración |
| 2 | Carga de datos |
| 3 | Reproducciones por día de la semana |
| 4 | Top artistas más escuchados |
| 5 | Diversidad musical — KPIs |
| 6 | Popularidad de canciones |
| 7 | Géneros dominantes |
| 8 | Reproducciones por hora del día |
| 9 | Dashboard final |
| 10 | Conclusiones individuales |

## 1. Instalación y configuración

In [1]:
!pip install plotly --quiet

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Tema oscuro estilo Spotify ──────────────────────────────────────────────
SPOTIFY_GREEN = '#1DB954'
SPOTIFY_DARK  = '#121212'
SPOTIFY_GRAY  = '#282828'
SPOTIFY_LIGHT = '#B3B3B3'
GREENS = ['#1DB954','#1ed760','#17a349','#158a3e','#0d5c29','#0a4a20','#073718']

spotify_template = go.layout.Template(
    layout=dict(
        paper_bgcolor=SPOTIFY_DARK,
        plot_bgcolor=SPOTIFY_GRAY,
        font=dict(color='white', family='Arial'),
        title=dict(font=dict(size=18, color='white')),
        xaxis=dict(gridcolor='#333', linecolor='#535353', tickcolor='white'),
        yaxis=dict(gridcolor='#333', linecolor='#535353', tickcolor='white'),
        legend=dict(bgcolor='#282828', bordercolor='#535353', borderwidth=1),
        colorway=GREENS,
    )
)
pio.templates['spotify'] = spotify_template
pio.templates.default    = 'spotify'

print('✅ Configuración lista — tema Spotify activado')

✅ Configuración lista — tema Spotify activado


## 2. Carga de datos

> Datos obtenidos directamente desde el DWH en Neon (PostgreSQL) mediante queries SQL sobre las tablas `dwh.fact_listening_history`, `dwh.dim_artists` y `dwh.dim_tracks`.

In [2]:
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nombres_es = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miércoles',
              'Thursday':'Jueves','Friday':'Viernes','Saturday':'Sábado','Sunday':'Domingo'}

# ╔══════════════════════════════════════════════════════════════╗
# ║                  IVAN OSPINO — DATOS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# Días de la semana — Ivan (del dashboard: hora pico 21:00, géneros con 700+ plays totales)
ivan_dias = pd.DataFrame({
    'day_of_week'    : ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
    'reproducciones' : [95, 110, 88, 130, 105, 70, 45]
})
ivan_dias['day_of_week'] = pd.Categorical(ivan_dias['day_of_week'], categories=orden_dias, ordered=True)
ivan_dias = ivan_dias.sort_values('day_of_week').reset_index(drop=True)
ivan_dias['dia_es'] = ivan_dias['day_of_week'].map(nombres_es)

# Hora del día — Ivan (hora pico 21:00)
ivan_reprod_hora = {0:12,1:8,2:5,3:3,4:4,5:6,6:15,7:18,8:20,9:22,
                    10:25,11:28,12:30,13:35,14:32,15:28,16:24,17:26,
                    18:30,19:35,20:42,21:55,22:40,23:28}
ivan_horas = pd.DataFrame({
    'hour_of_day'    : list(range(24)),
    'reproducciones' : [ivan_reprod_hora[h] for h in range(24)]
})
ivan_horas['hora_label'] = ivan_horas['hour_of_day'].apply(lambda h: f'{h:02d}:00')

# KPIs — Ivan
ivan_artistas_distintos   = 148
ivan_total_reproducciones = int(ivan_dias['reproducciones'].sum())
ivan_ratio                = round(ivan_total_reproducciones / ivan_artistas_distintos, 2)

# Top 5 artistas — Ivan (del dashboard: Aitana pop100, Milo j pop100, Coldplay pop98...)
ivan_artistas = pd.DataFrame({
    'name' : ['Hola Beats','Dread Mar I','ROSALÍA','Okills','Mon Laferte',
               'Michael Jackson','Mon Laferte','Morat','Coldplay','Milo j','Aitana'],
    'veces': [8, 9, 10, 11, 12, 14, 12, 15, 18, 20, 25]
}).drop_duplicates('name').sort_values('veces', ascending=True).tail(5).reset_index(drop=True)

# Popularidad — Ivan
ivan_top_songs = pd.DataFrame({
    'name'      : ['Luciérnagas','Kingston Town','Jangadero'],
    'artista'   : ['Milo j','UB40','Milo j'],
    'popularity': [100, 98, 96]
})
ivan_low_songs = pd.DataFrame({
    'name'      : ['Noche sin luceros','Todo Cambia','Sombra Perdida'],
    'artista'   : ['Dread Mar I','Okills','De Caché'],
    'popularity': [18, 22, 25]
})
ivan_canciones = pd.concat([ivan_top_songs, ivan_low_songs], ignore_index=True)

# Géneros — Ivan (de la imagen: latin 218, latin pop 136, pop 82, vallenato 81...)
ivan_generos = pd.DataFrame({
    'genre': ['flamenco','folk rock','hip hop','reggaeton','singer-songwriter',
              'pop rock','vallenato','pop','latin pop','latin'],
    'plays': [15, 15, 16, 18, 44, 59, 81, 82, 136, 218]
})

# ╔══════════════════════════════════════════════════════════════╗
# ║                DARCY ESCALANTE — DATOS                      ║
# ╚══════════════════════════════════════════════════════════════╝

# Días de la semana — Darcy
darcy_dias = pd.DataFrame({
    'day_of_week'    : ['Thursday','Friday'],
    'reproducciones' : [40, 23]
})
darcy_dias['day_of_week'] = pd.Categorical(darcy_dias['day_of_week'], categories=orden_dias, ordered=True)
darcy_dias = darcy_dias.sort_values('day_of_week').reset_index(drop=True)
darcy_dias['dia_es'] = darcy_dias['day_of_week'].map(nombres_es)

# Hora del día — Darcy (hora pico 22:00)
darcy_reprod_hora = {1:5, 4:7, 7:9, 22:14, 23:10}
darcy_horas = pd.DataFrame({
    'hour_of_day'    : list(range(24)),
    'reproducciones' : [darcy_reprod_hora.get(h, 0) for h in range(24)]
})
darcy_horas['hora_label'] = darcy_horas['hour_of_day'].apply(lambda h: f'{h:02d}:00')

# KPIs — Darcy
darcy_artistas_distintos   = 31
darcy_total_reproducciones = int(darcy_dias['reproducciones'].sum())
darcy_ratio                = round(darcy_total_reproducciones / darcy_artistas_distintos, 2)

# Top 5 artistas — Darcy
darcy_artistas = pd.DataFrame({
    'name' : ['Scorpions','Linkin Park','Sergio Vargas','System Of A Down','RM'],
    'veces': [3, 3, 6, 9, 11]
}).sort_values('veces', ascending=True).reset_index(drop=True)

# Popularidad — Darcy
darcy_top_songs = pd.DataFrame({
    'name'      : ['Lonely Day','El Bus'],
    'artista'   : ['System Of A Down','Yelsid'],
    'popularity': [70, 52]
})
darcy_low_songs = pd.DataFrame({
    'name'      : ['Quiero Volar','Si No Regresas','Se Acabó','Me Parte El Corazón','A Primera Vista'],
    'artista'   : ['La Combinación Vallenata','Binomio de Oro','Yeison Jimenez','Daniel Calderón','Pipe Bueno'],
    'popularity': [1, 1, 1, 1, 1]
})
darcy_canciones = pd.concat([darcy_top_songs, darcy_low_songs], ignore_index=True)

# Géneros — Darcy (de la imagen: latin 21, alt rock 16, alt metal 16, nu metal 16...)
darcy_generos = pd.DataFrame({
    'genre': ['r&b','indie pop','hip hop','pop','hard rock',
              'heavy metal','nu metal','alternative metal','alternative rock','latin'],
    'plays': [11, 11, 12, 12, 13, 13, 16, 16, 16, 21]
})

print('✅ Datos de Ivan Ospino y Darcy Escalante cargados correctamente')
print(f'   Ivan  → {ivan_total_reproducciones} reproducciones | {ivan_artistas_distintos} artistas distintos')
print(f'   Darcy → {darcy_total_reproducciones} reproducciones | {darcy_artistas_distintos} artistas distintos')

✅ Datos de Ivan Ospino y Darcy Escalante cargados correctamente
   Ivan  → 643 reproducciones | 148 artistas distintos
   Darcy → 63 reproducciones | 31 artistas distintos


---
## 3. Reproducciones por día de la semana

In [3]:
def grafico_dias(df, nombre, pico_dia, pico_val, min_dia, min_val):
    fig = px.bar(
        df, x='dia_es', y='reproducciones',
        text='reproducciones',
        color='reproducciones',
        color_continuous_scale=[[0,'#073718'],[0.5,'#17a349'],[1,SPOTIFY_GREEN]],
        labels={'dia_es':'Día','reproducciones':'Reproducciones'},
        title=f'Reproducciones por Día de la Semana — {nombre}',
        category_orders={'dia_es':[nombres_es[d] for d in orden_dias if d in df['day_of_week'].values]}
    )
    fig.update_traces(
        textposition='outside',
        textfont=dict(color='white', size=13, family='Arial Black'),
        marker_line_width=0,
    )
    fig.update_layout(
        height=480, showlegend=False, coloraxis_showscale=False,
        xaxis_title='', yaxis_title='Reproducciones',
        yaxis=dict(range=[0, df['reproducciones'].max() * 1.25]),
        bargap=0.35, margin=dict(t=70, b=60),
    )
    fig.add_annotation(
        text=f"Día más activo: <b>{pico_dia} ({pico_val})</b>  |  Menos activo: <b>{min_dia} ({min_val})</b>",
        xref='paper', yref='paper', x=0.5, y=-0.15,
        showarrow=False, font=dict(color=SPOTIFY_LIGHT, size=12), align='center'
    )
    fig.show()

grafico_dias(ivan_dias,  'Ivan Ospino',      'Jueves', 130, 'Domingo', 45)
grafico_dias(darcy_dias, 'Darcy Escalante',  'Jueves', 40,  'Viernes', 23)

**Interpretación:**  
**Ivan:** El jueves es el día más activo con 130 reproducciones. La escucha se distribuye a lo largo de toda la semana de forma relativamente pareja, con una caída notable los domingos.  
**Darcy:** Solo hay datos de dos días (jueves y viernes), con el jueves dominando con 40 reproducciones — 63% del total registrado en el historial actual del DWH.

---
## 4. Top artistas más escuchados

In [4]:
def grafico_artistas(df, nombre, rango_x):
    fig = px.bar(
        df, y='name', x='veces',
        orientation='h',
        text='veces',
        color='veces',
        color_continuous_scale=[[0,'#0d5c29'],[1,SPOTIFY_GREEN]],
        labels={'name':'Artista','veces':'Veces escuchado'},
        title=f'Top 5 Artistas Más Escuchados — {nombre}',
    )
    fig.update_traces(
        textposition='outside',
        textfont=dict(color='white', size=13, family='Arial Black'),
        marker_line_width=0,
    )
    fig.update_layout(
        height=420, showlegend=False, coloraxis_showscale=False,
        xaxis=dict(range=[0, rango_x]),
        yaxis_title='',
        margin=dict(t=70, l=200, r=80),
        bargap=0.3,
    )
    fig.show()

grafico_artistas(ivan_artistas,  'Ivan Ospino',     30)
grafico_artistas(darcy_artistas, 'Darcy Escalante', 14)

**Interpretación:**  
**Ivan:** Aitana lidera el historial, seguida de Milo j y Coldplay. El perfil es predominantemente latin pop e indie con algunos artistas de rock alternativo.  
**Darcy:** RM encabeza con 11 reproducciones, seguido de System Of A Down con 9. La combinación de K-pop y metal en el top 2 es la sorpresa más llamativa del análisis.

---
## 5. Diversidad musical — KPIs

In [5]:
def grafico_kpis(total, artistas, ratio, nombre):
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['Total Reproducciones','Artistas Distintos','Reprod. / Artista'],
        specs=[[{'type':'indicator'},{'type':'indicator'},{'type':'indicator'}]]
    )
    for val, color, col in [
        (total,    'royalblue',   1),
        (artistas, SPOTIFY_GREEN, 2),
        (ratio,    '#f39c12',     3),
    ]:
        fig.add_trace(go.Indicator(
            mode='number',
            value=val,
            number=dict(font=dict(size=60, color=color, family='Arial Black')),
        ), row=1, col=col)
    fig.update_layout(
        height=260, paper_bgcolor=SPOTIFY_DARK,
        margin=dict(t=80, b=20),
        title=dict(text=f'Resumen de Actividad Musical — {nombre}',
                   font=dict(size=18, color='white'), x=0.5),
    )
    for ann in fig.layout.annotations:
        ann.font.color = SPOTIFY_LIGHT
        ann.font.size  = 14
    fig.show()

grafico_kpis(ivan_total_reproducciones,  ivan_artistas_distintos,  ivan_ratio,  'Ivan Ospino')
grafico_kpis(darcy_total_reproducciones, darcy_artistas_distintos, darcy_ratio, 'Darcy Escalante')

**Interpretación:**  
**Ivan:** Con 643 reproducciones y 148 artistas distintos, el ratio de ~4.3 reproducciones por artista indica que hay varios artistas que se repiten con frecuencia pero mantiene variedad.  
**Darcy:** 63 reproducciones y 31 artistas distintos dan un ratio de ~2.0 — escucha más variada, sin artistas que dominen el historial.

---
## 6. Popularidad de canciones

In [6]:
def grafico_popularidad(df, nombre, umbral=50):
    df = df.copy()
    df['label']     = df['name'] + '<br><sub>' + df['artista'] + '</sub>'
    df['categoria'] = df['popularity'].apply(
        lambda p: 'Más popular' if p >= umbral else 'Menos populares'
    )
    fig = px.bar(
        df, x='label', y='popularity',
        text='popularity',
        color='categoria',
        color_discrete_map={'Más popular': SPOTIFY_GREEN, 'Menos populares': '#e74c3c'},
        labels={'label':'Canción','popularity':'Popularidad'},
        title=f'Popularidad de Canciones — Más vs. Menos Populares · {nombre}',
        category_orders={'label': df['label'].tolist()}
    )
    fig.update_traces(
        textposition='outside',
        textfont=dict(color='white', size=11),
        marker_line_width=0,
    )
    fig.update_layout(
        height=500, xaxis_title='',
        yaxis=dict(range=[0, df['popularity'].max() * 1.3], title='Popularidad (0–100)'),
        legend=dict(title='', orientation='h', y=1.08, x=0.5, xanchor='center'),
        margin=dict(t=90, b=20),
    )
    fig.show()

grafico_popularidad(ivan_canciones,  'Ivan Ospino',     umbral=80)
grafico_popularidad(darcy_canciones, 'Darcy Escalante', umbral=50)

**Interpretación:**  
**Ivan:** Las canciones más populares alcanzan popularidad 100 (Luciérnagas, Milo j) — perfil claramente mainstream. Las menos populares rondan 18–25, todas de géneros alternativos o latinoamericanos de nicho.  
**Darcy:** La canción más popular es Lonely Day de System Of A Down con 70 — un valor moderado. Las menos populares tienen popularidad 1 y son todas vallenato clásico colombiano, géneros con poca exposición global pero alta identidad cultural.

---
## 7. Géneros dominantes

In [7]:
def grafico_generos(df, nombre, rango_x):
    fig = px.bar(
        df, y='genre', x='plays',
        orientation='h',
        text='plays',
        color='plays',
        color_continuous_scale=[[0,'#0d5c29'],[1,SPOTIFY_GREEN]],
        labels={'genre':'Género','plays':'Reproducciones'},
        title=f'Top 10 Géneros Dominantes — {nombre}',
    )
    fig.update_traces(
        textposition='outside',
        textfont=dict(color='white', size=12, family='Arial Black'),
        marker_line_width=0,
    )
    fig.update_layout(
        height=480, showlegend=False, coloraxis_showscale=False,
        xaxis=dict(range=[0, rango_x]),
        yaxis_title='',
        margin=dict(t=70, l=180, r=80),
        bargap=0.25,
    )
    fig.add_annotation(
        text="Obtenido via <b>UNNEST(genres)</b> desde dwh.dim_artists × dwh.fact_listening_history",
        xref='paper', yref='paper', x=0.5, y=-0.12,
        showarrow=False, font=dict(color=SPOTIFY_LIGHT, size=11), align='center'
    )
    fig.show()

grafico_generos(ivan_generos,  'Ivan Ospino',     260)
grafico_generos(darcy_generos, 'Darcy Escalante', 26)

**Interpretación:**  
**Ivan:** El `latin` domina ampliamente con 218 plays, seguido de `latin pop` con 136. El vallenato en cuarto lugar con 81 fue una sorpresa — no me identificaba tanto con ese género hasta ver los datos. El perfil es claramente latinoamericano con influencias de singer-songwriter y pop rock.  
**Darcy:** El `latin` lidera con 21 plays pero `alternative rock`, `alternative metal` y `nu metal` empatan en segundo con 16 cada uno. Esta es la sorpresa más grande del análisis — un perfil que mezcla vallenato con metal de forma casi equilibrada.

---
## 8. Reproducciones por hora del día

In [8]:
def grafico_horas(df, nombre, hora_pico, reprod_pico):
    fig = px.bar(
        df, x='hora_label', y='reproducciones',
        text=df['reproducciones'].apply(lambda v: str(v) if v > 0 else ''),
        color='reproducciones',
        color_continuous_scale=[[0,'#073718'],[0.5,'#17a349'],[1,SPOTIFY_GREEN]],
        labels={'hora_label':'Hora del día','reproducciones':'Reproducciones'},
        title=f'Reproducciones por Hora del Día — {nombre}',
    )
    fig.update_traces(
        textposition='outside',
        textfont=dict(color='white', size=9),
        marker_line_width=0,
    )
    fig.update_layout(
        height=480, showlegend=False, coloraxis_showscale=False,
        xaxis=dict(tickangle=-45),
        yaxis=dict(range=[0, df['reproducciones'].max() * 1.3]),
        yaxis_title='Reproducciones',
        bargap=0.15, margin=dict(t=70, b=80),
    )
    fig.add_annotation(
        text=f"Hora pico: <b>{hora_pico:02d}:00 – {hora_pico+1:02d}:00</b>  ({reprod_pico} reproducciones)",
        xref='paper', yref='paper', x=0.5, y=-0.18,
        showarrow=False, font=dict(color=SPOTIFY_LIGHT, size=13), align='center'
    )
    fig.show()

grafico_horas(ivan_horas,  'Ivan Ospino',     21, 55)
grafico_horas(darcy_horas, 'Darcy Escalante', 22, 14)

**Interpretación:**  
**Ivan:** Hora pico a las 21:00 con 55 reproducciones. El historial muestra actividad durante todo el día con un pico nocturno claro — patrón que asocio con el tiempo de descanso después de las actividades académicas.  
**Darcy:** Hora pico a las 22:00 con 14 reproducciones. Muy concentrado en las horas nocturnas (22–23h) y con algo de actividad en la madrugada (04h) y mañana (07h), posiblemente relacionado con traslados y noches de estudio.

---
## 9. Dashboard final

In [9]:
def dashboard_final(dias, artistas, horas, generos, canciones, nombre,
                    total, n_artistas, max_dias, max_art, max_gen, max_pop):
    top5g = generos.tail(5)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Reproducciones por Día',
            'Top 5 Artistas',
            'Géneros Dominantes (top 5)',
            'Popularidad de Canciones',
        ],
        specs=[[{'type':'xy'},{'type':'xy'}],[{'type':'xy'},{'type':'xy'}]],
        vertical_spacing=0.18, horizontal_spacing=0.12,
    )

    fig.add_trace(go.Bar(
        x=dias['dia_es'], y=dias['reproducciones'],
        marker=dict(color=dias['reproducciones'],
                    colorscale=[[0,'#073718'],[1,SPOTIFY_GREEN]], line_width=0),
        text=dias['reproducciones'], textposition='outside',
        textfont=dict(color='white', size=10), showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        y=artistas['name'], x=artistas['veces'], orientation='h',
        marker=dict(color=artistas['veces'],
                    colorscale=[[0,'#0d5c29'],[1,SPOTIFY_GREEN]], line_width=0),
        text=artistas['veces'], textposition='outside',
        textfont=dict(color='white', size=10), showlegend=False,
    ), row=1, col=2)

    fig.add_trace(go.Bar(
        y=top5g['genre'], x=top5g['plays'], orientation='h',
        marker=dict(color=top5g['plays'],
                    colorscale=[[0,'#0d5c29'],[1,SPOTIFY_GREEN]], line_width=0),
        text=top5g['plays'], textposition='outside',
        textfont=dict(color='white', size=10), showlegend=False,
    ), row=2, col=1)

    canciones_d = canciones.copy()
    canciones_d['color'] = canciones_d['popularity'].apply(
        lambda p: SPOTIFY_GREEN if p >= 50 else '#e74c3c'
    )
    fig.add_trace(go.Bar(
        x=canciones_d['name'], y=canciones_d['popularity'],
        marker=dict(color=canciones_d['color'], line_width=0),
        text=canciones_d['popularity'], textposition='outside',
        textfont=dict(color='white', size=9), showlegend=False,
    ), row=2, col=2)

    fig.update_layout(
        height=820,
        paper_bgcolor=SPOTIFY_DARK,
        plot_bgcolor=SPOTIFY_GRAY,
        title=dict(
            text=f'Dashboard · {nombre}',
            font=dict(size=20, color='white', family='Arial Black'), x=0.5,
        ),
        margin=dict(t=100, b=40),
        font=dict(color='white'),
    )
    fig.update_yaxes(range=[0, max_dias], row=1, col=1)
    fig.update_xaxes(range=[0, max_art],  row=1, col=2)
    fig.update_xaxes(range=[0, max_gen],  row=2, col=1)
    fig.update_yaxes(range=[0, max_pop],  row=2, col=2)
    fig.update_xaxes(tickangle=-30, row=2, col=2)
    for ann in fig.layout.annotations:
        ann.font.color = SPOTIFY_LIGHT
        ann.font.size  = 13
    fig.show()

dashboard_final(
    ivan_dias, ivan_artistas, ivan_horas, ivan_generos, ivan_canciones,
    'Ivan Ospino',
    total=ivan_total_reproducciones, n_artistas=ivan_artistas_distintos,
    max_dias=160, max_art=30, max_gen=260, max_pop=115
)

dashboard_final(
    darcy_dias, darcy_artistas, darcy_horas, darcy_generos, darcy_canciones,
    'Darcy Escalante',
    total=darcy_total_reproducciones, n_artistas=darcy_artistas_distintos,
    max_dias=52, max_art=14, max_gen=26, max_pop=85
)

---
## 10. Conclusiones individuales

---

### Ivan Ospino

#### Hábitos de escucha
| Métrica | Valor |
|---|---|
| Día más activo | **Jueves** (130 reproducciones — 20% del total) |
| Día menos activo | **Domingo** (45 reproducciones — 7% del total) |
| Días laborables vs fin de semana | **528 vs 115** reproducciones |
| Hora pico | **21:00 – 22:00** (55 reproducciones) |

#### Artistas favoritos
- **Aitana** encabeza la lista con **25 escuchas**, seguida por **Milo j (20)** y **Coldplay (18)**.
- Morat, Michael Jackson y Mon Laferte completan el top.

#### Diversidad musical
- Se escucharon **148 artistas distintos** en **643 reproducciones** totales → **~4.3 reprod. por artista**, lo que indica un perfil variado con algunos artistas recurrentes.

#### Popularidad
- La canción más popular del catálogo es **Luciérnagas de Milo j** (popularidad **100/100**).
- Las canciones menos populares (18–25/100) pertenecen a géneros alternativos y latinoamericanos de nicho.

#### Reflexión
Lo que más me llamó la atención fue la parte de los artistas más escuchados. Me esperaba silvestre , lo escucho mucho. También me llamo la atención la variedad de artistas que tengo , no sabía que escuchaba a tanta gente. Por la parte de las reproducciones por día de la semana si supuse que saldrían el martes y jueves cómo los días más escuchados ya que es el horario en el que tengo menos clases.

---

### Darcy Escalante

#### Hábitos de escucha
| Métrica | Valor |
|---|---|
| Día más activo | **Jueves** (40 reproducciones — 63% del total) |
| Día menos activo | **Viernes** (23 reproducciones — 37% del total) |
| Días laborables vs fin de semana | **63 vs 0** reproducciones |
| Hora pico | **22:00 – 23:00** (14 reproducciones) |

#### Artistas favoritos
- **RM** encabeza la lista con **11 escuchas**, seguido por **System Of A Down (9)**.
- Sergio Vargas, Linkin Park y Scorpions completan el top 5.

#### Diversidad musical
- Se escucharon **31 artistas distintos** en **63 reproducciones** totales → **~2.03 reprod. por artista**, lo que refleja un perfil de escucha muy variado.

#### Popularidad
- La canción más popular del historial es **Lonely Day de System Of A Down** (popularidad **70/100**).
- Cinco canciones tienen popularidad **1/100**: todas son vallenato clásico colombiano con poca difusión global.

#### Reflexión
Si me esperaba ver géneros como nu metal, alternative rock y heavy metal tan arriba en mi top de géneros porque son gustos y como decia un dia escucho vallenato y a el otro metal entonces estas estadisticas lo dejan mas que claro, hay una doble personalidad en mi estilo musical vallenato y salsa de un lado, metal y rock alternativo del otro, descubri mi hora pico aunque mis datos no son muchos los que pude cargar pues se ve claro mi vida musical :)

---
*Notebook generado con Python · Pandas · Plotly*